# Ансамбли и случайный лес

CC BY 4.0.

Выполняйте ячейки сверху вниз. Сначала решите задачи, затем обратитесь к решениям.

## Чему научимся

- Объединять числовые прогнозы усреднением, а ответы-классы — голосованием.
- Объяснять, откуда берутся различия между деревьями случайного леса и зачем они нужны.
- Сравнивать дерево и лес на одних и тех же проверочных данных.

## План занятия

1. Объединим несколько прогнозов времени печати и посмотрим, когда это помогает.
2. Разберём, как из одной обучающей таблицы вырастить разные деревья.
3. Сравним дерево и случайный лес на знакомых учебных заказах.

## Могут ли несколько моделей быть лучше одной?

В [занятии 1.4](../04-decision-trees/content.md) мы предсказывали время 3D-печати с помощью дерева решений. В [занятии 1.7](../07-features-regularization/content.md) увидели, что подробное дерево может слишком сильно подстроиться под обучающие заказы. Стоит заменить часть заказов, и дерево иногда выбирает другие вопросы, а затем даёт другие прогнозы.

Попробуем поручить одну задачу нескольким моделям, а потом объединить их ответы. Такое объединение называется **ансамблем**. Модели внутри него могут быть разными, но в этом занятии все они будут деревьями решений.

Представим, что три дерева оценили время печати новой фигурки:

| Модель | Прогноз |
| --- | ---: |
| Первое дерево | 24 минуты |
| Второе дерево | 30 минут |
| Третье дерево | 36 минут |

Возьмём среднее:

$$
\frac{24+30+36}{3}=30\text{ минут}.
$$

Допустим, после печати выяснилось, что фигурка была готова за 31 минуту. Деревья ошиблись на 7, 1 и 5 минут, а ансамбль — на 1 минуту. Он оказался точнее двух деревьев и не хуже третьего. Это придуманный пример: он показывает способ объединения, но ещё не доказывает, что ансамбль будет хорош на других заказах.

В задаче **регрессии**, где ответ — число, усреднение естественно. В задаче **классификации**, где ответ — название класса, можно устроить голосование. Например, три дерева считают письмо спамом, а два — обычным письмом. По большинству голосов ансамбль выберет «спам». Усреднять сами названия классов нельзя.

В `RandomForestClassifier` из scikit-learn используется немного другой способ: деревья дают оценки вероятностей классов, лес усредняет эти оценки и выбирает класс с наибольшим результатом. Голосование помогает понять идею, но эти два способа могут дать разные ответы.

## Почему важны разные ошибки

Вернёмся к фигурке, которая печаталась 31 минуту. Если три дерева ответят 24, 25 и 26 минут, среднее составит 25 минут. Все недооценили время, поэтому усреднение не устранило общий промах.

А если сто раз скопировать дерево с прогнозом 24 минуты? Среднее ста одинаковых чисел тоже равно 24. Само количество моделей ничего не исправляет.

**Ансамблю полезны модели, которые замечают закономерности, но ошибаются по-разному.** Одно дерево иногда завышает прогноз, другое занижает; при усреднении такие отклонения могут частично компенсироваться. Случайные ответы без связи с данными не становятся хорошей моделью только от того, что их много.

Для глубоких деревьев этот приём особенно полезен: их ответы могут заметно зависеть от состава обучающих данных. Объединение разных деревьев часто делает прогноз менее чувствительным к такой замене. Пока нам достаточно этой идеи; математическое доказательство потребовало бы дополнительных знаний о случайных величинах.

## Бэггинг: каждому дереву своя таблица

Как получить разные деревья, если обучающая таблица одна? Подготовим для каждого дерева собственный набор строк. Для простоты представим пять заказов с именами А, Б, В, Г и Д.

Пять раз случайно выберем по одному заказу; у всех заказов одинаковые шансы. После каждого выбора возвращаем его в общий набор: один и тот же заказ разрешено выбрать снова. Возможный результат — **А, В, В, Д, А**. Строк по-прежнему пять, но заказ В встретился дважды, а Б и Г не попали в набор совсем. Копируем строки целиком, вместе с правильным временем печати; повтор не означает, что мы получили новое измерение.

Такой способ получения выборки называется **бутстрапом**, или выборкой **с возвращением**. В нашем варианте число выбираемых строк равно размеру исходной обучающей таблицы. Для следующего дерева повторим процедуру заново.

Обучим каждое дерево на его наборе и объединим прогнозы. Этот подход называется **бэггингом**. Деревья обучаются независимо друг от друга: второе не ждёт ошибок первого и не пытается специально их исправить. При этом они используют строки из общей таблицы, поэтому их ошибки всё же могут быть похожими.

## Случайный лес: добавим случайность при выборе признаков

Даже разные наборы строк могут привести к похожим деревьям. Например, если объём пластика хорошо объясняет время печати, многие деревья начнут с вопроса об объёме и продолжат строиться сходным образом.

Чтобы разнообразить деревья, при выборе **каждого следующего вопроса** будем случайно оставлять только часть признаков. Пусть доступны высота фигурки, число цветов и объём пластика. В одной вершине разрешим искать вопрос среди высоты и числа цветов. В другой — среди числа цветов и объёма. Набор выбирается заново в каждой вершине, а не один раз на всё дерево.

Среди разрешённых признаков дерево ищет хороший вопрос обычным способом. Случайно выбирается набор признаков, а не готовый ответ и не произвольный порог.

**Случайный лес** сочетает два приёма: каждое дерево получает выборку строк с возвращением, а при построении вопросов рассматривает случайную часть признаков. Для числового ответа лес усредняет прогнозы деревьев.



На схеме верхние стрелки показывают **обучение**, нижние — **получение прогноза**. Новый заказ поступает во все деревья с одинаковыми признаками. Числа 24, 30 и 36 взяты из нашего условного примера. В настоящем опыте ответы вычислят обученные модели.

## Сравним дерево и лес на Python

Используем 300 заказов из [занятия 1.7](../07-features-regularization/content.md). Данные синтетические: они придуманы для учебника и не описывают настоящий принтер. Предсказываем время печати в минутах по трём признакам, известным **до запуска печати**. Номер заказа и правильное время в признаки не включаем.

Запускайте [ноутбук](notebook.ipynb) из папки этого занятия в копии репозитория. Он читает таблицу соседнего занятия. Сначала загрузим данные:

In [ ]:
import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import KFold, cross_val_score
from sklearn.tree import DecisionTreeRegressor

data = pd.read_csv("../07-features-regularization/data/train.csv")
X = data[["height_cm", "colors", "plastic_cm3"]]
y = data["minutes"]


### Три проверки для каждой модели

Как в [занятии 1.8](../08-honest-evaluation/content.md), разделим строки на три части. Для каждой модели проведём **три проверки**: по очереди будем обучать её на двух частях, а проверять на третьей. Это выполнит `cross_val_score`.

In [ ]:
folds = KFold(n_splits=3, shuffle=True, random_state=42)
models = {
    "Дерево": DecisionTreeRegressor(random_state=0),
    "Лес": RandomForestRegressor(
        n_estimators=100, max_features=2,
        bootstrap=True, random_state=0,
    ),
}

for name, model in models.items():
    errors = -cross_val_score(
        model, X, y, cv=folds, scoring="neg_mean_absolute_error"
    )
    print(name, [round(error, 2) for error in errors],
          round(errors.mean(), 2))


Мера ошибки — знакомая **MAE**, средний модуль ошибки в минутах. Здесь библиотека возвращает её со знаком минус, поэтому перед вызовом стоит ещё один минус: в `errors` получаем обычные положительные ошибки.

`n_estimators=100` задаёт сто деревьев, `max_features=2` — выбор двух из трёх признаков при поиске очередного вопроса, `bootstrap=True` включает выбор строк с возвращением. `random_state` фиксирует случайные выборы, чтобы опыт можно было повторить. **Все бутстрап-выборки строятся только из обучающих строк текущей проверки.** Проверочная треть в них не попадает.

| Модель | Проверка 1 | Проверка 2 | Проверка 3 | Средняя MAE, мин |
| --- | ---: | ---: | ---: | ---: |
| Дерево | 3,02 | 2,97 | 2,69 | 2,89 |
| Лес из 100 деревьев | 2,76 | 2,48 | 2,55 | 2,60 |

На этих разбиениях лес ошибается меньше. Однако результат не означает, что лес всегда лучше любого дерева. Мы сравнили две конкретные модели на одном учебном наборе. Если по этим проверкам выбирать настройки, для независимой итоговой оценки понадобятся отдельные данные. Уже открытые итоговые ответы из занятия 1.8 для такой оценки нового выбора не используем.

## Что можно менять в лесе

У леса есть знакомые настройки деревьев: `max_depth` ограничивает глубину, а `min_samples_leaf` — минимальное число обучающих объектов в листе. В ансамбле часто используют глубокие деревья, но лес тоже может переобучаться. Ограничения подбирают по проверкам, а не по обучающей ошибке.

Есть и настройки самого ансамбля:

- **Число деревьев — `n_estimators`.** Больше деревьев обычно делает средний прогноз устойчивее, но требует больше памяти и времени. После некоторого момента улучшение становится небольшим. Уменьшение ошибки при каждом добавлении дерева не гарантировано.
- **Число признаков для вопроса — `max_features`.** Чем меньше доступных признаков, тем больше возможностей для различий между деревьями. Но слишком жёсткое ограничение может мешать им находить полезные вопросы. В нашем примере попробуйте значения 1, 2 и 3.

Сто деревьев не исправят неверные ответы в таблице или отсутствие важного признака. Кроме того, объяснить весь лес человеку труднее, чем пройти по нескольким вопросам одного небольшого дерева.

Бэггинг и случайный лес — не единственные способы объединять модели. В следующем занятии познакомимся с **бустингом**: там модели строят последовательно, чтобы новые помогали исправлять ошибки уже построенного ансамбля.

## Практика

1. Три дерева предсказали 26, 29 и 35 минут. Найдите средний прогноз. Если печать заняла 32 минуты, сравните ошибки отдельных деревьев и ансамбля. Обязан ли ансамбль быть точнее каждого дерева?
2. Для заказов А, Б, В, Г и Д составьте два возможных бутстрап-набора по пять строк. В каждом отметьте повторы и отсутствующие заказы. Объясните, почему нельзя просто раздать каждому дереву свою непересекающуюся часть таблицы и назвать это бутстрапом.
3. Воспроизведите таблицу из опыта. Затем изменяйте только `n_estimators`: возьмите 10 и 200 вместо 100. Сохраните разбиения и остальные настройки. Запишите среднюю MAE каждого варианта. Уменьшилась ли ошибка? По вашим наблюдениям, гарантирует ли увеличение числа деревьев заметный выигрыш?

<details><summary>Подсказка к практике</summary>

В первой задаче сначала сложите прогнозы. Во второй делайте пять выборов, каждый раз разрешая выбирать любой из исходных заказов. В третьей меняйте одну настройку, чтобы понимать причину изменения результата.

</details>


## Итоги

Ансамбль объединяет ответы нескольких моделей. В случайном лесе различия между деревьями создают выбором строк с возвращением и случайным выбором признаков при каждом разбиении. Усреднение помогает сгладить часть ошибок, но пользу леса нужно проверять на данных, которые деревья не видели при обучении.

## Источники и дальнейшее чтение

Основной ориентир для темы — [«Ансамбли в машинном обучении» в хендбуке Яндекса](https://education.yandex.ru/handbook/ml/article/ansambli-v-mashinnom-obuchenii). Там можно подробнее прочитать о бэггинге и случайном лесе, а позднее вернуться к математическому объяснению. Примеры с заказами, схема и упражнения в этом занятии подготовлены для нашего учебника.

Для работы с кодом пригодятся описание [ансамблей в scikit-learn](https://scikit-learn.org/stable/modules/ensemble.html#forest) и справочники [RandomForestRegressor](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestRegressor.html) и [RandomForestClassifier](https://scikit-learn.org/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html).

## Итоговый опрос

Сначала ответьте самостоятельно, затем раскройте подсказку и решение.

### 1. Три дерева предсказали 18, 24 и 30 минут. Как ответит ансамбль с усреднением?

<details><summary>Подсказка</summary>

Каждое дерево вносит одинаковый вклад.

</details>


### 2. Из пяти деревьев четыре выбрали «спам», одно — «обычное письмо». Каков ответ при голосовании по большинству?

<details><summary>Подсказка</summary>

Сравните число голосов за каждый класс.

</details>


### 3. Почему сто одинаковых копий дерева не исправят его прогноз?

<details><summary>Подсказка</summary>

Чему равно среднее ста одинаковых чисел?

</details>


### 4. Может ли бутстрап-набор из пяти строк содержать заказ А трижды и ни разу не содержать заказ Б?

<details><summary>Подсказка</summary>

Возвращается ли выбранный заказ в общий набор?

</details>


### 5. Когда в случайном лесе выбирают случайный набор признаков для вопроса?

<details><summary>Подсказка</summary>

Может ли у соседних вершин одного дерева быть разный набор разрешённых признаков?

</details>


### 6. Если вершине разрешены высота и число цветов, выбирается ли порог высоты наугад?

<details><summary>Подсказка</summary>

Различайте выбор доступных признаков и поиск вопроса.

</details>


### 7. Нужно ли второму дереву леса дождаться ошибок первого, чтобы начать обучение?

<details><summary>Подсказка</summary>

Вспомните различие леса и бустинга.

</details>


### 8. Можно ли построить бутстрап-наборы из всех 300 строк, а затем проверить лес на ста строках из той же таблицы?

<details><summary>Подсказка</summary>

Могли ли проверочные ответы попасть в обучение хотя бы одного дерева?

</details>


### 9. MAE дерева — 2,89, леса — 2,60 минуты. Что показал наш опыт?

<details><summary>Подсказка</summary>

Сформулируйте вывод именно о проведённом опыте.

</details>


### 10. Деревьев стало вдвое больше, а ошибка почти не изменилась. Программа сломана?

<details><summary>Подсказка</summary>

Добавляют ли новые деревья новые сведения о мире?

</details>



## Решения задач

### Задача 1

1. Среднее равно 30 минутам. Ошибки деревьев — 6, 3 и 3 минуты, ансамбля — 2 минуты. Здесь среднее точнее всех, но это не обязательно: при истинном времени 29 минут второе дерево было бы точным, а среднее ошиблось бы на минуту.
2. Например, А, А, В, Д, Д и Б, В, Г, Г, А. В первом наборе нет Б и Г, во втором — Д. При раздаче непересекающихся частей строка досталась бы только одному дереву и не могла бы повториться внутри его набора.
3. При указанных разбиениях и настройках средние MAE для 10, 100 и 200 деревьев — примерно 2,61, 2,60 и 2,59 минуты. В этом опыте выигрыш невелик, хотя число деревьев выросло в двадцать раз. Небольшое улучшение или ухудшение на этих проверках не даёт гарантии для других данных. После такого подбора это результаты выбора настройки, а не независимая итоговая оценка.

### Задача 2

24 минуты: сумма 72 делится на 3. Усредняются прогнозы для одного и того же заказа.

### Задача 3

«Спам»: этот класс получил четыре голоса из пяти. Здесь задано именно голосование, а не усреднение вероятностей.

### Задача 4

Копии дают тот же ответ и допускают ту же ошибку: среднее одинаковых чисел равно каждому из них.

### Задача 5

Да, например: А, А, В, А, Д. Выбираем с возвращением, поэтому повторения и пропуски допустимы.

### Задача 6

При поиске разбиения в каждой вершине заново. Одно дерево может использовать разные признаки в разных вопросах.

### Задача 7

Нет. Среди разрешённых признаков дерево ищет хороший вопрос по обучающим данным.

### Задача 8

Нет. Дерево леса обучается независимо от ответов других деревьев; исправление ошибок предшественников — идея бустинга.

### Задача 9

Нельзя: лес может уже знать проверочные ответы. Сначала отделяем проверочную часть, затем строим бутстрап-наборы только из обучающих строк.

### Задача 10

Выбранный лес точнее на этих разбиениях. Обобщать результат на любые модели и данные нельзя.

### Задача 11

Не обязательно. Прогноз может меняться всё слабее; новые деревья требуют вычислений, но не добавляют отсутствующих сведений о мире.